# Load Dataset

In [2]:
################################################################################
# Load dataset and split it into training and test set (multi-class)
################################################################################

import pandas as pd
import os
from tabulate import tabulate

dataset_name = "cic-iot"
sample_size = 100000

# Load dataset
df        = pd.read_csv(os.getcwd() + '/data/sample-multiclass.csv',  low_memory=False)
train_df  = pd.read_csv(os.getcwd() + '/data/train-multiclass.csv',   low_memory=False)
test_df   = pd.read_csv(os.getcwd() + '/data/test-multiclass.csv',    low_memory=False)

# Keep original multi-class labels — NO binary merging

train_dfs, test_dfs = {}, {}
for label, group in df.groupby('label'):
    train_part = group.sample(frac=0.8, random_state=42)
    test_part  = group.drop(train_part.index)
    train_dfs[label] = train_part
    test_dfs[label]  = test_part

df_train = pd.concat(train_dfs.values())
df_test  = pd.concat(test_dfs.values())

X_train = train_df.drop(columns=['label'])
y_train = train_df['label']
X_test  = test_df.drop(columns=['label'])
y_test  = test_df['label']

# Table: class | total | train | test
table_data = [[lbl, len(group), len(train_dfs[lbl]), len(test_dfs[lbl])]
              for lbl, group in df.groupby('label')]
print(tabulate(table_data, headers=["Class", "Total", "Train", "Test"], tablefmt="grid"))


+--------------------+---------+---------+--------+
| Class              |   Total |   Train |   Test |
+====================+=========+=========+========+
| BenignTraffic      |    2348 |    1878 |    470 |
+--------------------+---------+---------+--------+
| DDoS-Flood         |    2348 |    1878 |    470 |
+--------------------+---------+---------+--------+
| DDoS-Fragmentation |    2163 |    1730 |    433 |
+--------------------+---------+---------+--------+
| DNS_Spoofing       |     405 |     324 |     81 |
+--------------------+---------+---------+--------+
| DoS-Flood          |    2348 |    1878 |    470 |
+--------------------+---------+---------+--------+
| MITM-ArpSpoofing   |     653 |     522 |    131 |
+--------------------+---------+---------+--------+
| Mirai              |    2348 |    1878 |    470 |
+--------------------+---------+---------+--------+
| Recon              |     705 |     564 |    141 |
+--------------------+---------+---------+--------+


# ML Baseline (Multi-Class)

In [3]:
################################################################################
# Decision Tree and Random Forest (multi-class)
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import os
import time

os.makedirs("results/ml", exist_ok=True)

for Model, fname in [(DecisionTreeClassifier, "result-dt-multiclass-100000.txt"),
                     (RandomForestClassifier,  "result-rf-multiclass-100000.txt")]:
    model = Model(random_state=42)
    model.fit(X_train, y_train)
    start = time.time()
    y_pred = model.predict(X_test)
    elapsed = time.time() - start
    report = classification_report(y_true=y_test, y_pred=y_pred, digits=4)
    matrix = confusion_matrix(y_test, y_pred)
    print(f"=== {Model.__name__} ({elapsed:.4f}s) ===")
    print(report)
    with open(f"results/ml/{fname}", "w") as f:
        f.write(f"Classification Report\n{report}\n\nConfusion Matrix\n{matrix}\n")


=== DecisionTreeClassifier (0.0007s) ===
                    precision    recall  f1-score   support

     BenignTraffic     0.9163    0.8851    0.9004       470
        DDoS-Flood     0.9979    0.9936    0.9957       470
DDoS-Fragmentation     0.9954    0.9954    0.9954       433
      DNS_Spoofing     0.6744    0.7160    0.6946        81
         DoS-Flood     0.9979    1.0000    0.9989       470
  MITM-ArpSpoofing     0.6889    0.7099    0.6992       131
             Mirai     0.9957    0.9936    0.9947       470
             Recon     0.7467    0.7943    0.7698       141

          accuracy                         0.9430      2666
         macro avg     0.8766    0.8860    0.8811      2666
      weighted avg     0.9444    0.9430    0.9436      2666

=== RandomForestClassifier (0.0114s) ===
                    precision    recall  f1-score   support

     BenignTraffic     0.8577    0.9617    0.9067       470
        DDoS-Flood     1.0000    0.9957    0.9979       470
DDoS-Fragmenta

# Vector Store — Per-Class Representative Samples

In [4]:
################################################################################
# Build class_entries dict with representative samples per class
################################################################################

import json
import numpy as np
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'mps'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 64}
)

feature_cols = X_train.columns.tolist()
n_results = 10
max_embed_per_class = 100   # cap to avoid slow startup

class_entries = {}
for label in tqdm(df['label'].unique(), desc="Building class entries"):
    class_df = train_dfs[label].drop(columns=['label'])
    sample_df = class_df.sample(n=min(max_embed_per_class, len(class_df)), random_state=42)
    docs = [str(row.tolist()) for _, row in sample_df.iterrows()]
    # Embed docs
    vecs = np.array(embeddings.embed_documents(docs))
    mean_vec = vecs.mean(axis=0)
    # Cosine similarities to mean
    norms = np.linalg.norm(vecs, axis=1) * np.linalg.norm(mean_vec)
    sims = (vecs @ mean_vec) / np.where(norms == 0, 1e-9, norms)
    top_idx = np.argsort(sims)[::-1][:n_results]
    top_rows = sample_df.iloc[top_idx]
    # Transpose to feature-keyed dict
    entries = {}
    for col in feature_cols:
        entries[col] = top_rows[col].tolist()
    class_entries[label] = entries

print("class_entries keys:", list(class_entries.keys()))


Building class entries: 100%|██████████| 8/8 [00:46<00:00,  5.75s/it]

class_entries keys: ['DDoS-Flood', 'DoS-Flood', 'Mirai', 'BenignTraffic', 'DDoS-Fragmentation', 'Recon', 'MITM-ArpSpoofing', 'DNS_Spoofing']


# Tool Definition

In [5]:
################################################################################
# Tool: evaluate_rule (multi-class aware)
################################################################################

from sklearn.metrics import classification_report
from tqdm import tqdm
import operator
from typing import Annotated
from langchain_core.tools import tool

show_progress = False
operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq,
              '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

@tool
def evaluate_rule(
    feature_name: Annotated[str, "Feature name"],
    value:        Annotated[str, "Value"],
    op:           Annotated[str, "Operator"],
    target_class: Annotated[str, "Target class name"]
) -> float:
    """Evaluate rule on training set. Returns macro F1 for target_class vs all others."""
    try:
        value = float(value)
    except ValueError:
        pass
    if op not in operations:
        raise ValueError(f"Unsupported operator: {op}")
    y_true, y_pred = [], []
    for lbl, df_cls in train_dfs.items():
        df_feat = df_cls.drop(columns=['label'])
        for i in tqdm(range(len(df_feat)), disable=not show_progress,
                      desc=f"Eval {lbl[:12]}..."):
            y_true.append("target" if lbl == target_class else "other")
            y_pred.append("target" if operations[op](df_feat.iloc[i][feature_name], value) else "other")
    report = classification_report(y_true, y_pred, digits=4, output_dict=True, zero_division=0)
    return report['macro avg']['f1-score']


# Prompt Template

In [17]:
################################################################################
# Prompt Template
################################################################################

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = ("system",
"""You are a network security analyst specialising in IoT intrusion detection.
You are given labelled network traffic data with named features and numeric values.
Your task is to generate exactly {k} deterministic threshold rules to identify '{target_class}' traffic.

Rules must:
- Use ONLY inequality operators: '>', '<', '>=', '<='
- NEVER use '==' or '!=' — these are forbidden for numeric features
- Reference only feature names present in the data
- Be range-based thresholds that generalise beyond the exact sample values shown
- You MUST make exactly {k} tool calls — one per rule, no more, no less.

Good rule example: Header_Length < 100
Bad rule example: Header_Length == 54.0  ← FORBIDDEN"""
)

human_message = ("user",
"""Analyze the following network data and generate rules for the top {k} important features \
to identify '{target_class}' entries.

Target Class Entries:
```{target_entries}```

Other Class Entries (sample):
```{other_entries}```"""
)

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder("msgs")
])


# LLM Setup

In [18]:
################################################################################
# LLM Setup and Tool Binding
################################################################################

import os
import dotenv
# from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

dotenv.load_dotenv(os.getcwd() + '/../.env')

# model_name = "gpt-4o"
model_name = "claude-haiku-4-5-20251001"
# llm = ChatOpenAI(model=model_name, temperature=0.1)
llm = ChatAnthropic(model=model_name, temperature=0.1)

llm_with_tool = llm.bind_tools([evaluate_rule])


# Feedback Loop (Per Class)

In [20]:
################################################################################
# Per-class feedback loop — improved
# Persist learned rules to disk for downstream evaluation
################################################################################

import json
import os
from langchain_core.messages import HumanMessage

chain = prompt | llm_with_tool

CLASS_CONFIG = {
    'DDoS-Flood':        {'max_rounds': 15, 'k': 5},
    'DoS-Flood':         {'max_rounds': 12, 'k': 7},  # oscillating — more k helps
    'Mirai':             {'max_rounds': 15, 'k': 5},
    'DDoS-Fragmentation':{'max_rounds': 15, 'k': 5},
    'Recon':             {'max_rounds': 12, 'k': 7},  # hard class — more rules
    'MITM-ArpSpoofing':  {'max_rounds': 12, 'k': 7},
    'DNS_Spoofing':      {'max_rounds': 15, 'k': 7},  # weakest — needs both
}

# n_repetitions  = 8        # more rounds — DNS_Spoofing needed 4 just to recover
# k              = 5
patience       = 4        # early stop if no improvement for this many rounds
show_progress  = False
class_rules    = {}

os.makedirs("results/llm", exist_ok=True)
rules_output_path = "results/llm/class_rules-multiclass-100000.json"

non_benign_classes = [lbl for lbl in df['label'].unique() if lbl != 'BenignTraffic']

# Build other_entries: sample from ALL non-target classes (not just BenignTraffic)
# Cap at 3 rows per class to keep prompt size manageable
def build_other_entries(target_class, class_entries, rows_per_class=3):
    other = {}
    for cls, entries in class_entries.items():
        if cls != target_class:
            other[cls] = list(entries)[:rows_per_class]
    return other

for target_class in non_benign_classes:
    print(f"\n=== Target class: {target_class} ===")

    cfg        = CLASS_CONFIG[target_class]
    max_rounds = cfg['max_rounds']
    k          = cfg['k']

    n               = 0
    no_improve      = 0
    best_mean_f1    = 0.0
    best_tool_calls = []
    train_f1_scores = []
    msgs            = []

    target_entries = json.dumps(class_entries[target_class])
    other_entries  = json.dumps(build_other_entries(target_class, class_entries))

    while n < max_rounds and no_improve < patience:
        ai_msg = chain.invoke({
            "k": k, "target_class": target_class,
            "target_entries": target_entries,
            "other_entries": other_entries,
            "msgs": msgs
        })

        # Invoke tools and collect per-rule scores
        tool_msgs   = []
        rule_scores = []
        for tool_call in ai_msg.tool_calls:
            tool_msg = evaluate_rule.invoke(tool_call)
            tool_msgs.append(tool_msg)
            try:
                rule_scores.append(float(tool_msg.content))
            except (ValueError, TypeError):
                rule_scores.append(0.0)

        mean_f1 = sum(rule_scores) / len(rule_scores) if rule_scores else 0.0

        # Best-state preservation — only keep rules if this round improved
        if mean_f1 > best_mean_f1:
            best_mean_f1 = mean_f1
            best_tool_calls = list(ai_msg.tool_calls)
            no_improve = 0
        else:
            no_improve += 1

        # Per-rule score feedback — tell the model exactly which rules scored what
        rule_feedback_lines = []
        for i, (tc, score) in enumerate(zip(ai_msg.tool_calls, rule_scores)):
            args = tc["args"]
            rule_feedback_lines.append(
                f"  Rule {i+1}: {args['feature_name']} {args['op']} {args['value']} "
                f"-> F1={score:.4f} ({'KEEP' if score >= mean_f1 else 'REVISE'})"
            )
        rule_feedback = "\n".join(rule_feedback_lines)

        human_msg = HumanMessage(
            f"Round {n+1} results (best so far: {best_mean_f1:.4f}):\n"
            f"{rule_feedback}\n\n"
            f"Mean F1 this round: {mean_f1:.4f}. "
            f"Revise rules marked REVISE by trying different threshold values or a different feature. "
            f"Keep rules marked KEEP unchanged. "
            f"Generate exactly {k} rules for '{target_class}' and make exactly {k} tool calls."
        )

        msgs.extend([ai_msg, *tool_msgs, human_msg])
        train_f1_scores.append(mean_f1)
        n += 1

        usage_info = ai_msg.response_metadata.get("usage", {})
        token_usage = {
            "completion_tokens": usage_info.get("output_tokens", 0),
            "prompt_tokens": usage_info.get("input_tokens", 0),
            "total_tokens": usage_info.get("input_tokens", 0) + usage_info.get("output_tokens", 0)
        }
        print(f"  Round: {n}  Mean F1: {mean_f1:.4f}  Best: {best_mean_f1:.4f}  "
              f"No-improve: {no_improve}  Tokens: {token_usage}")
        
        # After collecting rule_scores, validate operators
        for tc in ai_msg.tool_calls:
            if tc["args"]["op"] in ["==", "!="]:
                print(f"  WARNING: equality operator detected in rule — {tc['args']}")

    # Use best rules, not last-round rules; store JSON-serializable structure
    class_rules[target_class] = [{"args": tc["args"]} for tc in best_tool_calls]
    print(f"  Train F1 history: {train_f1_scores}")
    print(f"  Final best F1: {best_mean_f1:.4f}  (stopped after {n} rounds)")


with open(rules_output_path, "w") as f:
    json.dump(class_rules, f, indent=2)

print("\nDone. Classes with rules:", list(class_rules.keys()))
print(f"Saved class rules to: {rules_output_path}")


=== Target class: DDoS-Flood ===
  Round: 1  Mean F1: 0.4269  Best: 0.4269  No-improve: 0  Tokens: {'completion_tokens': 710, 'prompt_tokens': 3371, 'total_tokens': 4081}
  Round: 2  Mean F1: 0.5901  Best: 0.5901  No-improve: 0  Tokens: {'completion_tokens': 548, 'prompt_tokens': 4481, 'total_tokens': 5029}
  Round: 3  Mean F1: 0.6197  Best: 0.6197  No-improve: 0  Tokens: {'completion_tokens': 553, 'prompt_tokens': 5429, 'total_tokens': 5982}
  Round: 4  Mean F1: 0.5085  Best: 0.6197  No-improve: 1  Tokens: {'completion_tokens': 557, 'prompt_tokens': 6385, 'total_tokens': 6942}
  Round: 5  Mean F1: 0.6808  Best: 0.6808  No-improve: 0  Tokens: {'completion_tokens': 555, 'prompt_tokens': 7347, 'total_tokens': 7902}
  Round: 6  Mean F1: 0.6880  Best: 0.6880  No-improve: 0  Tokens: {'completion_tokens': 552, 'prompt_tokens': 8304, 'total_tokens': 8856}
  Round: 7  Mean F1: 0.6909  Best: 0.6909  No-improve: 0  Tokens: {'completion_tokens': 547, 'prompt_tokens': 9258, 'total_tokens': 9805}


In [ ]:
with open("results/llm/class_rules-multiclass-100000.json") as f:
    rules = json.load(f)
print(json.dumps(rules['DDoS-Flood'], indent=2))

[
  {
    "args": {
      "feature_name": "Max",
      "op": "==",
      "value": "54.0",
      "target_class": "DDoS-Flood"
    }
  },
  {
    "args": {
      "feature_name": "Header_Length",
      "op": "==",
      "value": "54.0",
      "target_class": "DDoS-Flood"
    }
  },
  {
    "args": {
      "feature_name": "Magnitue",
      "op": "==",
      "value": "10.392304845413264",
      "target_class": "DDoS-Flood"
    }
  },
  {
    "args": {
      "feature_name": "AVG",
      "op": "==",
      "value": "54.0",
      "target_class": "DDoS-Flood"
    }
  },
  {
    "args": {
      "feature_name": "Tot sum",
      "op": "==",
      "value": "567.0",
      "target_class": "DDoS-Flood"
    }
  }
]


# Evaluate All Rules on Test Set (Multi-Class)

In [25]:
# Check 1: do the feature names actually exist in X_test?
for tc in rules['DDoS-Flood']:
    feat = tc['args']['feature_name']
    print(f"'{feat}' in X_test.columns: {feat in X_test.columns}")

'AVG' in X_test.columns: True
'Radius' in X_test.columns: True
'Std' in X_test.columns: True
'Tot sum' in X_test.columns: True
'Max' in X_test.columns: True


In [27]:
for i, (_, row) in enumerate(ddos_test_rows.iterrows()):
    scores = {}
    for cls, tool_calls in class_rules.items():
        if len(tool_calls) == 0:
            scores[cls] = 0.0
            continue
        count = sum(
            1 for tc in tool_calls
            if tc['args']['op'] in operations
            and tc['args']['feature_name'] in row.index
            and operations[tc['args']['op']](
                row[tc['args']['feature_name']],
                float(tc['args']['value'])
            )
        )
        scores[cls] = count / len(tool_calls)
    print(f"Row {i}: {scores} → predicted: {max(scores, key=scores.get)}")

Row 0: {'DDoS-Flood': 0.6, 'DoS-Flood': 1.0, 'Mirai': 0.4, 'DDoS-Fragmentation': 0.0, 'Recon': 0.2857142857142857, 'MITM-ArpSpoofing': 0.0, 'DNS_Spoofing': 0.2857142857142857} → predicted: DoS-Flood
Row 1: {'DDoS-Flood': 1.0, 'DoS-Flood': 1.0, 'Mirai': 0.6, 'DDoS-Fragmentation': 0.0, 'Recon': 0.2857142857142857, 'MITM-ArpSpoofing': 0.0, 'DNS_Spoofing': 0.2857142857142857} → predicted: DDoS-Flood
Row 2: {'DDoS-Flood': 1.0, 'DoS-Flood': 0.7142857142857143, 'Mirai': 0.6, 'DDoS-Fragmentation': 0.0, 'Recon': 0.14285714285714285, 'MITM-ArpSpoofing': 0.0, 'DNS_Spoofing': 0.0} → predicted: DDoS-Flood
Row 3: {'DDoS-Flood': 1.0, 'DoS-Flood': 1.0, 'Mirai': 0.4, 'DDoS-Fragmentation': 0.0, 'Recon': 0.2857142857142857, 'MITM-ArpSpoofing': 0.0, 'DNS_Spoofing': 0.2857142857142857} → predicted: DDoS-Flood
Row 4: {'DDoS-Flood': 1.0, 'DoS-Flood': 0.7142857142857143, 'Mirai': 0.6, 'DDoS-Fragmentation': 0.0, 'Recon': 0.14285714285714285, 'MITM-ArpSpoofing': 0.2857142857142857, 'DNS_Spoofing': 0.14285714285

In [26]:
################################################################################
# Evaluate generated rules on test set (multi-class)
# Load learned rules from disk instead of in-memory state
################################################################################

import json
import operator
import os
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq,
              '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

os.makedirs("results/llm", exist_ok=True)
rules_input_path = "results/llm/class_rules-multiclass-100000.json"

with open(rules_input_path, "r") as f:
    class_rules = json.load(f)

# Sanity check before running 2666 predictions
assert len(class_rules) == 7, f"Expected 7 classes, got {len(class_rules)}"
for cls, rules in class_rules.items():
    assert len(rules) > 0, f"Class {cls} has no rules"
print(f"Loaded {len(class_rules)} classes, rule counts: { {c: len(r) for c, r in class_rules.items()} }")

def predict_multiclass(row, class_rules):
    scores = {}
    for cls, tool_calls in class_rules.items():
        if len(tool_calls) == 0:
            scores[cls] = 0.0
            continue
        count = 0
        for tc in tool_calls:
            args = tc["args"]
            op, feat, val = args["op"], args["feature_name"], args["value"]
            try:
                val = float(val)
            except ValueError:
                pass
            if op in operations and feat in row.index:
                if operations[op](row[feat], val):
                    count += 1
        # Normalise: proportion of rules that fired, not raw count
        scores[cls] = count / len(tool_calls)

    max_score = max(scores.values()) if scores else 0.0
    if max_score == 0.0:
        return 'BenignTraffic'

    winners = [cls for cls, s in scores.items() if s == max_score]
    return winners[0] if len(winners) == 1 else 'BenignTraffic'

y_pred_llm, y_true_llm = [], []
for i in tqdm(range(len(X_test)), desc="Evaluating multiclass rules"):
    row = X_test.iloc[i]
    y_true_llm.append(y_test.iloc[i])
    y_pred_llm.append(predict_multiclass(row, class_rules))

report = classification_report(y_true_llm, y_pred_llm, digits=4)
matrix = confusion_matrix(y_true_llm, y_pred_llm)
print(report)
with open("results/llm/result-llm-multiclass-100000.txt", "w") as f:
    f.write(f"Classification Report\n{report}\n\nConfusion Matrix\n{matrix}\n")

Loaded 7 classes, rule counts: {'DDoS-Flood': 5, 'DoS-Flood': 7, 'Mirai': 5, 'DDoS-Fragmentation': 5, 'Recon': 7, 'MITM-ArpSpoofing': 7, 'DNS_Spoofing': 7}


Evaluating multiclass rules: 100%|██████████| 2666/2666 [00:00<00:00, 15478.76it/s]


                    precision    recall  f1-score   support

     BenignTraffic     0.2865    0.3426    0.3120       470
        DDoS-Flood     0.5163    0.4383    0.4741       470
DDoS-Fragmentation     0.7482    0.9400    0.8332       433
      DNS_Spoofing     0.1065    0.2222    0.1440        81
         DoS-Flood     0.6370    0.1979    0.3019       470
  MITM-ArpSpoofing     0.3540    0.3053    0.3279       131
             Mirai     0.8068    0.9064    0.8537       470
             Recon     0.3024    0.4397    0.3584       141

          accuracy                         0.5300      2666
         macro avg     0.4697    0.4740    0.4506      2666
      weighted avg     0.5542    0.5300    0.5171      2666



# Efficiency Comparison

In [15]:
################################################################################
# Efficiency Comparison
################################################################################

import time
from tabulate import tabulate

# Retrain DT and RF for timing
model_dt = DecisionTreeClassifier(random_state=42)
model_rf = RandomForestClassifier(random_state=42)
model_dt.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

elapsed_dt, elapsed_rf, elapsed_llm = [], [], []
n_samples = min(1000, len(X_test))   # cap to avoid slow LLM rules

for i in range(n_samples):
    row = X_test.iloc[[i]]

    t0 = time.time()
    model_dt.predict(row)
    elapsed_dt.append(time.time() - t0)
    
    t0 = time.time()
    model_rf.predict(row)
    elapsed_rf.append(time.time() - t0)
    
    t0 = time.time()
    predict_multiclass(X_test.iloc[i], class_rules)
    elapsed_llm.append(time.time() - t0)

rows = [
    ["Decision Tree", f"{sum(elapsed_dt)/n_samples*1e3:.4f} ms"],
    ["Random Forest", f"{sum(elapsed_rf)/n_samples*1e3:.4f} ms"],
    ["LLM Rules",     f"{sum(elapsed_llm)/n_samples*1e3:.4f} ms"],
]
print(tabulate(rows, headers=["Model", "Avg Inference Time"], tablefmt="grid"))


+---------------+----------------------+
| Model         | Avg Inference Time   |
+===============+======================+
| Decision Tree | 0.2739 ms            |
+---------------+----------------------+
| Random Forest | 1.8946 ms            |
+---------------+----------------------+
| LLM Rules     | 0.0606 ms            |
+---------------+----------------------+
